# 🔍 Project 10 — CritiqueLoop: Self-Reflective Agent

**Core Concept:** Generate → Evaluate → Critique → Improve loop

### Architecture
Draft → Evaluator → Score >= 8? → Done
                  → Score < 8  → Critic → Rewriter → Repeat

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.2 MB/s eta 0:00:00


API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "Key"

All Setup In One Block

In [4]:
import os
import json
from datetime import datetime, timezone
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Generator ────────────────────────────────────────────────
class Generator:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a skilled writer and problem solver.
Generate a clear, comprehensive, and well-structured response.
Be specific, accurate, and helpful."""),
            ("human", "Task: {task}\n\nPrevious critique (if any): {critique}\n\nGenerate response:")
        ])
        self.chain = self.prompt | self.llm

    def generate(self, task: str, critique: str = "None") -> str:
        logger.info("Generator creating response...")
        response = self.chain.invoke({
            "task": task,
            "critique": critique
        })
        return response.content


# ── Evaluator ────────────────────────────────────────────────
class Evaluator:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a strict quality evaluator.
Evaluate the response on these criteria:
1. Accuracy — is it factually correct?
2. Completeness — does it fully address the task?
3. Clarity — is it clear and well-structured?
4. Usefulness — is it actionable and helpful?

Respond ONLY in this exact JSON format:
{{"score": 7.5, "accuracy": 8, "completeness": 7, "clarity": 8, "usefulness": 7, "summary": "one line summary of quality"}}"""),
            ("human", "Task: {task}\n\nResponse to evaluate:\n{response}\n\nEvaluate:")
        ])
        self.chain = self.prompt | self.llm

    def evaluate(self, task: str, response: str) -> dict:
        logger.info("Evaluator scoring response...")
        result = self.chain.invoke({
            "task": task,
            "response": response
        })

        raw = result.content.strip()
        if "```" in raw:
            import re
            raw = re.sub(r"```(?:json)?", "", raw).strip()

        try:
            data = json.loads(raw)
            return data
        except:
            return {
                "score": 5.0,
                "accuracy": 5,
                "completeness": 5,
                "clarity": 5,
                "usefulness": 5,
                "summary": "Could not parse evaluation"
            }


# ── Critic ───────────────────────────────────────────────────
class Critic:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a harsh but constructive critic.
Identify specific weaknesses in the response.
Be precise about what is missing, wrong, or unclear.
Give actionable improvement suggestions.
Keep critique under 100 words."""),
            ("human", "Task: {task}\n\nResponse:\n{response}\n\nEvaluation scores: {scores}\n\nProvide specific critique:")
        ])
        self.chain = self.prompt | self.llm

    def critique(self, task: str, response: str, scores: dict) -> str:
        logger.info("Critic analyzing response...")
        result = self.chain.invoke({
            "task": task,
            "response": response,
            "scores": json.dumps(scores)
        })
        return result.content


# ── CritiqueLoop Agent ───────────────────────────────────────
class CritiqueLoopAgent:
    def __init__(self, quality_threshold: float = 8.0, max_iterations: int = 3):
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            api_key=os.environ["GROQ_API_KEY"]
        )
        self.quality_threshold = quality_threshold
        self.max_iterations = max_iterations
        self.generator = Generator(self.llm)
        self.evaluator = Evaluator(self.llm)
        self.critic = Critic(self.llm)
        logger.info(f"CritiqueLoop initialized | Threshold: {quality_threshold} | Max iterations: {max_iterations}")

    def run(self, task: str) -> dict:
        logger.info(f"Starting CritiqueLoop for: {task[:60]}")

        iteration_history = []
        critique = "None"
        final_response = None
        final_score = 0

        for iteration in range(1, self.max_iterations + 1):
            logger.info(f"Iteration {iteration}/{self.max_iterations}")

            response = self.generator.generate(task, critique)
            scores = self.evaluator.evaluate(task, response)
            score = scores.get("score", 0)

            logger.info(f"Score: {score}/10 | Summary: {scores.get('summary', 'N/A')}")

            iteration_data = {
                "iteration": iteration,
                "response": response,
                "scores": scores,
                "score": score,
                "critique": critique
            }

            if score >= self.quality_threshold:
                iteration_data["status"] = "accepted"
                iteration_history.append(iteration_data)
                final_response = response
                final_score = score
                logger.info(f"Quality threshold met at iteration {iteration}")
                break

            critique = self.critic.critique(task, response, scores)
            iteration_data["critique_for_next"] = critique
            iteration_data["status"] = "improved"
            iteration_history.append(iteration_data)
            final_response = response
            final_score = score

            logger.info(f"Critique generated — moving to iteration {iteration + 1}")

        return {
            "task": task,
            "final_response": final_response,
            "final_score": final_score,
            "iterations_used": len(iteration_history),
            "threshold": self.quality_threshold,
            "iteration_history": iteration_history,
            "improved": len(iteration_history) > 1
        }

    def display_result(self, result: dict):
        print("\n" + "="*60)
        print("CRITIQUELOOP RESULT")
        print("="*60)
        print(f"Task            : {result['task'][:80]}")
        print(f"Iterations Used : {result['iterations_used']}/{self.max_iterations}")
        print(f"Final Score     : {result['final_score']}/10")
        print(f"Quality Met     : {result['final_score'] >= self.quality_threshold}")
        print(f"Was Improved    : {result['improved']}")

        print("\n--- ITERATION HISTORY ---")
        for item in result['iteration_history']:
            print(f"\n[Iteration {item['iteration']}] Score: {item['score']}/10 | Status: {item['status']}")
            print(f"Scores: {item['scores'].get('summary', 'N/A')}")
            if item.get('critique_for_next') and item['critique_for_next'] != "None":
                print(f"Critique: {item['critique_for_next'][:150]}")

        print(f"\n--- FINAL RESPONSE ---")
        print(result['final_response'][:500])
        print("="*60)

agent = CritiqueLoopAgent(quality_threshold=8.0, max_iterations=3)
print("CritiqueLoop agent ready")

06:08:48 | INFO | CritiqueLoop initialized | Threshold: 8.0 | Max iterations: 3
CritiqueLoop agent ready


Test 1: Simple Task

In [5]:
result = agent.run(
    "Explain what an API is in simple terms for a beginner"
)
agent.display_result(result)

06:08:56 | INFO | Starting CritiqueLoop for: Explain what an API is in simple terms for a beginner
06:08:56 | INFO | Iteration 1/3
06:08:56 | INFO | Generator creating response...
06:08:58 | INFO | Evaluator scoring response...
06:08:59 | INFO | Score: 9.0/10 | Summary: The response accurately and clearly explains what an API is and how it works in simple terms for a beginner.
06:08:59 | INFO | Quality threshold met at iteration 1

CRITIQUELOOP RESULT
Task            : Explain what an API is in simple terms for a beginner
Iterations Used : 1/3
Final Score     : 9.0/10
Quality Met     : True
Was Improved    : False

--- ITERATION HISTORY ---

[Iteration 1] Score: 9.0/10 | Status: accepted
Scores: The response accurately and clearly explains what an API is and how it works in simple terms for a beginner.

--- FINAL RESPONSE ---
**What is an API?**

Imagine you're at a restaurant and you want to order food. You can't just walk into the kitchen and start making your own food because that's

Test 2: Technical Task

In [6]:
result = agent.run(
    "Write a Python function to find all prime numbers up to n using the Sieve of Eratosthenes"
)
agent.display_result(result)

06:09:03 | INFO | Starting CritiqueLoop for: Write a Python function to find all prime numbers up to n us
06:09:04 | INFO | Iteration 1/3
06:09:04 | INFO | Generator creating response...
06:09:05 | INFO | Evaluator scoring response...
06:09:05 | INFO | Score: 9.5/10 | Summary: High-quality implementation and explanation of the Sieve of Eratosthenes algorithm in Python
06:09:05 | INFO | Quality threshold met at iteration 1

CRITIQUELOOP RESULT
Task            : Write a Python function to find all prime numbers up to n using the Sieve of Era
Iterations Used : 1/3
Final Score     : 9.5/10
Quality Met     : True
Was Improved    : False

--- ITERATION HISTORY ---

[Iteration 1] Score: 9.5/10 | Status: accepted
Scores: High-quality implementation and explanation of the Sieve of Eratosthenes algorithm in Python

--- FINAL RESPONSE ---
**Sieve of Eratosthenes Implementation in Python**

The Sieve of Eratosthenes is an ancient algorithm used to find all prime numbers up to a given number `n`. H

Test 3: Complex Task

In [7]:
result = agent.run(
    "Create a comprehensive study plan for someone learning machine learning from scratch in 6 months"
)
agent.display_result(result)

06:09:11 | INFO | Starting CritiqueLoop for: Create a comprehensive study plan for someone learning machi
06:09:11 | INFO | Iteration 1/3
06:09:11 | INFO | Generator creating response...
06:09:15 | INFO | Evaluator scoring response...
06:09:16 | INFO | Score: 9.2/10 | Summary: Comprehensive and well-structured 6-month study plan for learning machine learning from scratch, covering mathematical foundations, programming basics, machine learning fundamentals, deep learning, and advanced topics.
06:09:16 | INFO | Quality threshold met at iteration 1

CRITIQUELOOP RESULT
Task            : Create a comprehensive study plan for someone learning machine learning from scr
Iterations Used : 1/3
Final Score     : 9.2/10
Quality Met     : True
Was Improved    : False

--- ITERATION HISTORY ---

[Iteration 1] Score: 9.2/10 | Status: accepted
Scores: Comprehensive and well-structured 6-month study plan for learning machine learning from scratch, covering mathematical foundations, programming basics,

Score Comparison

In [8]:
print("========== SCORE IMPROVEMENT ANALYSIS ==========\n")

tasks = [
    "Explain recursion in programming",
    "Write a function to reverse a linked list in Python",
    "Describe the CAP theorem in distributed systems"
]

print(f"{'Task':<50} {'Iter':<6} {'Score':<8} {'Improved'}")
print("-" * 75)

for task in tasks:
    result = agent.run(task)
    improved = "Yes" if result['improved'] else "No"
    print(f"{task[:48]:<50} {result['iterations_used']:<6} {result['final_score']:<8} {improved}")

========== SCORE IMPROVEMENT ANALYSIS ==========

Task                                               Iter   Score    Improved
---------------------------------------------------------------------------
06:09:19 | INFO | Starting CritiqueLoop for: Explain recursion in programming
06:09:19 | INFO | Iteration 1/3
06:09:19 | INFO | Generator creating response...
06:09:21 | INFO | Evaluator scoring response...
06:09:22 | INFO | Score: 9.2/10 | Summary: The response provides a comprehensive and well-structured explanation of recursion in programming, covering its definition, how it works, examples, advantages, disadvantages, best practices, and common use cases.
06:09:22 | INFO | Quality threshold met at iteration 1
Explain recursion in programming                   1      9.2      No
06:09:22 | INFO | Starting CritiqueLoop for: Write a function to reverse a linked list in Python
06:09:22 | INFO | Iteration 1/3
06:09:22 | INFO | Generator creating response...
06:09:23 | INFO | Evaluator scor

Project Summary

In [ ]:
print("========== CRITIQUELOOP SUMMARY ==========\n")
print("Project         : CritiqueLoop — Self-Reflective Agent")
print("Author          : K Murali Krishna")
print("Model           : Groq LLaMA-3.3-70b-versatile")
print(f"Quality Threshold: {agent.quality_threshold}/10")
print(f"Max Iterations  : {agent.max_iterations}")
print("\nComponents:")
print("  ✓ Generator  — creates initial and improved responses")
print("  ✓ Evaluator  — scores response on 4 dimensions")
print("  ✓ Critic     — identifies specific weaknesses")
print("  ✓ Loop       — iterates until quality threshold met")
print("\nEvaluation Dimensions:")
print("  ✓ Accuracy      — factual correctness")
print("  ✓ Completeness  — fully addresses the task")
print("  ✓ Clarity       — clear and well-structured")
print("  ✓ Usefulness    — actionable and helpful")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Self-reflection pattern")
print("  ✓ Quality-gated output")
print("  ✓ Iterative improvement")
print("  ✓ Structured evaluation scoring")